In [1]:
import pandas as pd
import os
import random
import chardet
import requests
import json
import numpy as np
from urllib3.util.retry import Retry
import openai
from requests.adapters import HTTPAdapter
import os
import pickle
import matplotlib.pyplot as plt
# from utils.read_group_danmu import read_group_danmu

In [2]:
import chardet
import os
import pandas as pd
import numpy as np
import random
import re

# 自定义转换函数
def convert_to_seconds(time_str):
    # 处理 bytes 类型
    if isinstance(time_str, bytes):
        time_str = time_str.decode('utf-8')
    try:
        return pd.to_timedelta(time_str).total_seconds()
    except Exception:
        return np.nan
    
def read_group_danmu_fixwindow(file_dir, file, n_danmu_per_group, movie_time_name='movie_time', danmu_name='danmu'):
    danmu_dict = {}
    with open(os.path.join(file_dir, file), 'rb') as f:
        encoding_type = chardet.detect(f.read())['encoding']

    print(file)
    try:
        df = pd.read_csv(os.path.join(file_dir, file), encoding='gbk')
        print('use gbk encoding')
    except:
        df = pd.read_csv(os.path.join(file_dir, file), encoding=encoding_type)
        print('use %s encoding' % encoding_type)

    print(file, 'danmu number:', len(df))
    print(df[:5])
    # 初始化结果列表
    result = []
    time_ranges = []

    df[movie_time_name] = df[movie_time_name].apply(
        lambda x: x.decode('utf-8') if isinstance(x, bytes) else str(x)
    )

    first_element = df[movie_time_name].iloc[0]
    if isinstance(first_element, bytes):
        first_element = first_element.decode('utf-8')
    print(first_element)
    test_numeric = pd.to_numeric(pd.Series([first_element]), errors='coerce')
    is_number = test_numeric.notna().iloc[0]

    if is_number:
        df[movie_time_name] = pd.to_numeric(df[movie_time_name], errors='coerce')
        df = df[np.isfinite(df[movie_time_name])]
        df.reset_index(drop=True, inplace=True)
    else:
        df[movie_time_name] = df[movie_time_name].apply(convert_to_seconds)
        df = df.dropna(subset=[movie_time_name])
        df[movie_time_name] = df[movie_time_name].astype(int)

    df[movie_time_name] = df[movie_time_name].apply(lambda x: float(x))
    df = df.sort_values(by=movie_time_name)
    
    time_values = df[movie_time_name].astype(int)
    time_min = int(time_values.min())
    if time_min < 0:
        time_min = 0
    # 如果最大值导致范围过大，用第二大的
    candidate_max = int(time_values.max())
    if candidate_max - time_min > 43200:  # 超过12小时不合理
        # 取第二大的唯一值
        unique_sorted = np.sort(time_values.unique())
        candidate_max = int(unique_sorted[-2])
        print(f"最大值异常，使用第二大值: {candidate_max}")

    time_max = candidate_max
    
    # 按秒分组，聚合弹幕为列表
    grouped = df.groupby(df[movie_time_name].astype(int))[danmu_name].apply(list)
    
    # 创建完整时间索引，缺失的秒填充空列表
    all_seconds = range(time_min, time_max + 1)
    grouped = grouped.reindex(all_seconds, fill_value=[])
    
    print(f"总秒数: {len(grouped)} (从 {time_min} 到 {time_max})")
    
    # 遍历每一秒（包括空的）
    for time_int, texts in grouped.items():
        if len(texts) == 0:
            # 这一秒没有弹幕
            result.append('')  # 空字符串
            time_ranges.append((float(time_int), float(time_int)))  # 时间范围就是这一秒本身
        elif len(texts) > n_danmu_per_group:
            # 弹幕太多，随机采样
            sampled_indices = random.sample(range(len(texts)), n_danmu_per_group)
            sampled_texts = [str(texts[i]) for i in sampled_indices]
            
            result.append('\n'.join(sampled_texts))
            # 计算采样弹幕的实际时间范围
            times_for_this_second = df[df[movie_time_name].astype(int) == time_int][movie_time_name].tolist()
            sampled_times = [times_for_this_second[i] for i in sampled_indices]
            time_ranges.append((min(sampled_times), max(sampled_times)))
        else:
            # 弹幕数量在限制内，全部保留
            sampled_texts = [str(texts[i]) for i in range(len(texts))]
            
            result.append('\n'.join(sampled_texts))
            times_for_this_second = df[df[movie_time_name].astype(int) == time_int][movie_time_name].tolist()
            time_ranges.append((min(times_for_this_second), max(times_for_this_second)))

    danmu_dict[file] = result
    danmu_dict[file+'_time_range'] = time_ranges
    return danmu_dict

In [ ]:
# with open('results/failed_counts_prompt_v4.pkl', 'rb') as f:
#     failed_counts_dict = pickle.load(f)

In [ ]:
import requests
import json
import numpy as np
from urllib3.util.retry import Retry
import openai
from requests.adapters import HTTPAdapter
import os
import pickle

print_every = 400
for file_name in ['Fantastic_Beasts_And_Where_To_Find_Them']:
    print('\n\n', file_name)
    file = file_name + '.csv'
    with open(r'\\10.16.57.94\dataset1\xinke\Danmu\DyEmo-submit\DyEmo-FullMovies\Processed_danmu\danmu_%s_downsample.pkl' % file_name, 'rb') as f:
        danmu_dict = pickle.load(f)

    datadir = 'results/prompt_v4'

    url = "https://api.gpt.ge/v1/chat/completions"
    headers = {
        "Content-Type": "application/json",
        "Authorization": "" # Please fill in your api key here
    }

    # 创建一个session
    session = requests.Session()

    # 定义重试策略，包括处理SSL错误
    retries = Retry(
        total=10,  # 总重试次数
        backoff_factor=1,  # 重试间隔时间
        status_forcelist=[500, 502, 503, 504],  # 需要重试的状态码
        raise_on_status=False,  # 不因状态码引发异常
        raise_on_redirect=False  # 不因重定向引发异常
    )

    # 将重试策略应用到session
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("http://", adapter)
    session.mount("https://", adapter)

    openai.requestssession = session

    print(file)
        
    danmu = danmu_dict[file]
    time_ranges = danmu_dict[file+'_time_range']
    
    max_interval = 0
    for j in range(len(time_ranges)):
        max_interval = np.maximum(max_interval, time_ranges[j][1] - time_ranges[j][0])
    if max_interval >= 45:
        print('max interval over 45, continue')
        continue

    if not os.path.exists(os.path.join(datadir, file[:-4])):
        os.makedirs(os.path.join(datadir, file[:-4]))
    
    for i in range(len(danmu)):
        if len(danmu[i]) == 0:
            continue
            
        data = {
            "model": "gpt-3.5-turbo",
            "messages": [
                {
                    "role": "user",
                    "content": "任务：请综合以下视频弹幕推测观众在观看对应视频时的情绪感受。对高兴、惊讶、悲伤、愤怒、厌恶、恐惧六种情绪类别进行评价，评分为0到7之间的连续取值（可精确到小数点后一位），0表示完全没有，7表示非常强烈。\n弹幕（两条弹幕之间以换行分隔）：%s\n请严格按以下格式给出评分结果（无需返回额外的评论）：高兴: [评分]; 惊讶: [评分]; 悲伤: [评分]; 愤怒: [评分]; 厌恶: [评分]; 恐惧: [评分]" %  danmu[i]
                }
            ],
            "temperature": 0.0
        }
        
        try:
            response = session.post(url, headers=headers, json=data)
            response.raise_for_status()  
            
            with open(os.path.join(datadir, file[:-4], "%.3f_%.3f.json" % (time_ranges[i][0], time_ranges[i][1])), "w", encoding="utf-8") as f:
                json.dump(response.json(), f, ensure_ascii=False, indent=4)
            
            print(time_ranges[i][0], time_ranges[i][1], response.json()['choices'][0]['message']['content'])

        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")




 Fantastic_Beasts_And_Where_To_Find_Them
Fantastic_Beasts_And_Where_To_Find_Them.csv
719.037 719.87 高兴: 6.5; 惊讶: 5.0; 悲伤: 1.0; 愤怒: 0.0; 厌恶: 0.0; 恐惧: 0.0
720.04 720.742 高兴: 6.5; 惊讶: 5.0; 悲伤: 0.0; 愤怒: 0.0; 厌恶: 0.0; 恐惧: 0.0
721.063 721.957 高兴: 6.5; 惊讶: 5.0; 悲伤: 0.0; 愤怒: 0.0; 厌恶: 0.0; 恐惧: 0.0
722.306 722.915 高兴: 6.5; 惊讶: 5.0; 悲伤: 0.0; 愤怒: 0.0; 厌恶: 0.0; 恐惧: 0.0
723.007 723.506 高兴: 6.5; 惊讶: 2.0; 悲伤: 0.0; 愤怒: 0.0; 厌恶: 0.0; 恐惧: 0.0
724.053 724.998 高兴: 6.5; 惊讶: 5.0; 悲伤: 0.0; 愤怒: 0.0; 厌恶: 0.0; 恐惧: 0.0
725.058 725.752 高兴: 5.0; 惊讶: 4.0; 悲伤: 0.0; 愤怒: 0.0; 厌恶: 0.0; 恐惧: 0.0
726.284 726.782 高兴: 5.0; 惊讶: 6.5; 悲伤: 0.0; 愤怒: 0.0; 厌恶: 0.0; 恐惧: 0.0
727.238 727.997 高兴: 5.0; 惊讶: 6.0; 悲伤: 0.0; 愤怒: 0.0; 厌恶: 0.0; 恐惧: 0.0
728.109 728.957 高兴: 6.5; 惊讶: 5.0; 悲伤: 0.0; 愤怒: 0.0; 厌恶: 0.0; 恐惧: 0.0
729.106 729.319 高兴: 5.0; 惊讶: 4.5; 悲伤: 0.0; 愤怒: 0.0; 厌恶: 0.0; 恐惧: 0.0
730.06 730.873 高兴: 6.5; 惊讶: 5.0; 悲伤: 0.0; 愤怒: 0.0; 厌恶: 0.0; 恐惧: 0.0
731.217 731.892 高兴: 5.0; 惊讶: 6.5; 悲伤: 3.0; 愤怒: 0.0; 厌恶: 0.0; 恐惧: 0.0
732.133 732.501 高兴: